<h1 style="text-align: center; color: #007acc; background-color: #f0f8ff; padding: 20px; border-radius: 10px; border-left: 5px solid #007acc;">🍽️ MiseEnPlace | Metrics & KPIs</h1>

## Overview
This notebook transforms the clean dataset into actionable insights
for a gastronomy investment committee.

**<span style="color: #28a745; font-weight: bold;">Input:</span>** `../data/processed/michelin_cleaned.csv`  
**<span style="color: #dc3545; font-weight: bold;">Question:</span>** What defines a 3-Star Michelin restaurant —
and what does that mean for investors?

---

In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/michelin_cleaned.csv')
df_facilities = pd.read_csv('../data/processed/michelin_facilities_exploded.csv')

order_award = ['3 Stars', '2 Stars', '1 Star', 'Bib Gourmand', 'Selected Restaurants']

print(f"Main dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Facilities dataset: {df_facilities.shape[0]} rows x {df_facilities.shape[1]} columns")


Main dataset: 18908 rows x 8 columns
Facilities dataset: 45017 rows x 3 columns


<h2 style="color: #28a745;">1. Facilities & Services Analysis</h2>

The first step is understanding the null distribution across award categories.
A null value in `FacilitiesAndServices` means the restaurant has no registered
facilities — this absence itself is an insight.

In [2]:
nulls_by_award=df.groupby('Award')['FacilitiesAndServices'].apply( lambda x:x.isnull().sum()).reindex(order_award)

total_by_award=df['Award'].value_counts().reindex(order_award)

null_pct=(nulls_by_award/total_by_award*100).round(2)

summary=pd.DataFrame({
    'Total Restaurants': total_by_award,
    'No Facilities': nulls_by_award,
    'No Facilities (%)': null_pct
})

summary

,Total Restaurants,No Facilities,No Facilities (%)
Award,,,
3 Stars,154,0,0.00
2 Stars,536,5,0.93
1 Star,3145,100,3.18
Bib Gourmand,3566,249,6.98
Selected Restaurants,11507,768,6.67


<h3 style="color: #28a745;">Finding: Facilities as a Quality Signal</h3>

<table style="border-collapse: collapse; width: 100%;">
<tr style="background-color: #4e4e4eff;">
<th style="border: 1px solid #ddd; padding: 8px;">Award</th>
<th style="border: 1px solid #ddd; padding: 8px;">Total</th>
<th style="border: 1px solid #ddd; padding: 8px;">No Facilities</th>
<th style="border: 1px solid #ddd; padding: 8px;">No Facilities (%)</th>
</tr>
<tr>
<td style="border: 1px solid #ddd; padding: 8px;"><span style="color: #dc3545;">3 Stars</span></td>
<td style="border: 1px solid #ddd; padding: 8px;">154</td>
<td style="border: 1px solid #ddd; padding: 8px;">0</td>
<td style="border: 1px solid #ddd; padding: 8px;">0.0%</td>
</tr>
<tr style="background-color: #4e4e4eff;">
<td style="border: 1px solid #ddd; padding: 8px;"><span style="color: #ffc107;">2 Stars</span></td>
<td style="border: 1px solid #ddd; padding: 8px;">536</td>
<td style="border: 1px solid #ddd; padding: 8px;">5</td>
<td style="border: 1px solid #ddd; padding: 8px;">0.9%</td>
</tr>
<tr>
<td style="border: 1px solid #ddd; padding: 8px;"><span style="color: #28a745;">1 Star</span></td>
<td style="border: 1px solid #ddd; padding: 8px;">3,145</td>
<td style="border: 1px solid #ddd; padding: 8px;">100</td>
<td style="border: 1px solid #ddd; padding: 8px;">3.2%</td>
</tr>
<tr style="background-color: #4e4e4eff;">
<td style="border: 1px solid #ddd; padding: 8px;"><span style="color: #007bff;">Bib Gourmand</span></td>
<td style="border: 1px solid #ddd; padding: 8px;">3,566</td>
<td style="border: 1px solid #ddd; padding: 8px;">249</td>
<td style="border: 1px solid #ddd; padding: 8px;">7.0%</td>
</tr>
<tr>
<td style="border: 1px solid #ddd; padding: 8px;"><span style="color: #6c757d;">Selected Restaurants</span></td>
<td style="border: 1px solid #ddd; padding: 8px;">11,507</td>
<td style="border: 1px solid #ddd; padding: 8px;">768</td>
<td style="border: 1px solid #ddd; padding: 8px;">6.7%</td>
</tr>
</table>

**Key insight:** 100% of 3-Star Michelin restaurants have registered
facilities and services. The percentage of restaurants without facilities
increases consistently as award level decreases.

This suggests that investing in facilities and services is not optional
for top-tier restaurants — it is a prerequisite for excellence.

For investors: **any restaurant without registered facilities
should be treated as a red flag.**

---

<h3 style="color: #28a745;">1.2 Facility Frequency by Award Category</h3>

Now that we know facilities are a prerequisite for top-tier restaurants,
the next question is: **which specific facilities matter most?**

We calculate the penetration rate of each facility in 3-Star restaurants —
how many of the 154 restaurants have each service registered.

In [21]:
# Step 1 — Top facilities in 3 Stars restaurants
total_3_stars = df[df['Award']=='3 Stars'].shape[0]
facilities_3_stars=df_facilities[df_facilities['Award']=='3 Stars']['FacilitiesAndServices'].value_counts()

penetration_3_stars=(facilities_3_stars/total_3_stars *100).round(2)

print("=== Top Facilities in 3-Star Restaurants (%) ===")
print(f"Base: {total_3_stars} restaurants\n")
penetration_3_stars

=== Top Facilities in 3-Star Restaurants (%) ===
Base: 154 restaurants



FacilitiesAndServices
Air conditioning         93.51
Interesting wine list    84.42
Wheelchair access        56.49
Car park                 48.05
Valet parking            24.68
Garden or park           18.18
Great view               16.23
Counter dining           12.99
Terrace                  12.99
Notable sake list         5.19
Shoes must be removed     3.25
Cash only                 0.65
Brunch                    0.65
Name: count, dtype: float64

In [19]:
# Step 2 — Compare top 5 facilities across all award categories
top_5_facilities=penetration_3_stars.head(5).index.to_list()

comparison=df_facilities[
    df_facilities['FacilitiesAndServices'].isin(top_5_facilities)
].groupby(['Award','FacilitiesAndServices']).size().unstack(fill_value=0)

comparison=comparison.div(total_by_award,axis=0).mul(100).round(2)
comparison=comparison.reindex(order_award)[top_5_facilities]

print("=== Top 5 Facilities — Penetration Rate by Award Category (%) ===")
comparison

=== Top 5 Facilities — Penetration Rate by Award Category (%) ===


FacilitiesAndServices,Air conditioning,Interesting wine list,Wheelchair access,Car park,Valet parking
Award,,,,,
3 Stars,93.51,84.42,56.49,48.05,24.68
2 Stars,87.31,70.34,49.44,41.98,18.28
1 Star,75.10,37.20,37.23,38.41,8.43
Bib Gourmand,61.19,6.67,21.73,23.95,2.30
Selected Restaurants,63.51,12.88,30.80,29.93,5.79


<h3 style="color: #28a745;">Finding: The Facility Blueprint of a 3-Star Restaurant</h3>

Comparing the top 5 facilities across all award categories reveals
a clear pattern: the higher the award, the higher the facility penetration.

<table style="border-collapse: collapse; width: 100%;">
<tr style="background-color: #4e4e4eff;">
<th style="border: 1px solid #ddd; padding: 8px;">Facility</th>
<th style="border: 1px solid #ddd; padding: 8px;"><span style="color: #dc3545;">3 Stars</span></th>
<th style="border: 1px solid #ddd; padding: 8px;"><span style="color: #ffc107;">2 Stars</span></th>
<th style="border: 1px solid #ddd; padding: 8px;"><span style="color: #28a745;">1 Star</span></th>
<th style="border: 1px solid #ddd; padding: 8px;"><span style="color: #007bff;">Bib Gourmand</span></th>
<th style="border: 1px solid #ddd; padding: 8px;"><span style="color: #6c757d;">Selected</span></th>
</tr>
<tr>
<td style="border: 1px solid #ddd; padding: 8px;">Air conditioning</td>
<td style="border: 1px solid #ddd; padding: 8px;">93.5%</td>
<td style="border: 1px solid #ddd; padding: 8px;">87.3%</td>
<td style="border: 1px solid #ddd; padding: 8px;">75.1%</td>
<td style="border: 1px solid #ddd; padding: 8px;">61.2%</td>
<td style="border: 1px solid #ddd; padding: 8px;">63.5%</td>
</tr>
<tr style="background-color: #4e4e4eff;">
<td style="border: 1px solid #ddd; padding: 8px;">Interesting wine list</td>
<td style="border: 1px solid #ddd; padding: 8px;">84.4%</td>
<td style="border: 1px solid #ddd; padding: 8px;">70.3%</td>
<td style="border: 1px solid #ddd; padding: 8px;">37.2%</td>
<td style="border: 1px solid #ddd; padding: 8px;">6.7%</td>
<td style="border: 1px solid #ddd; padding: 8px;">12.9%</td>
</tr>
<tr>
<td style="border: 1px solid #ddd; padding: 8px;">Wheelchair access</td>
<td style="border: 1px solid #ddd; padding: 8px;">56.5%</td>
<td style="border: 1px solid #ddd; padding: 8px;">49.4%</td>
<td style="border: 1px solid #ddd; padding: 8px;">37.2%</td>
<td style="border: 1px solid #ddd; padding: 8px;">21.7%</td>
<td style="border: 1px solid #ddd; padding: 8px;">30.8%</td>
</tr>
<tr style="background-color: #4e4e4eff;">
<td style="border: 1px solid #ddd; padding: 8px;">Car park</td>
<td style="border: 1px solid #ddd; padding: 8px;">48.1%</td>
<td style="border: 1px solid #ddd; padding: 8px;">42.0%</td>
<td style="border: 1px solid #ddd; padding: 8px;">38.4%</td>
<td style="border: 1px solid #ddd; padding: 8px;">24.0%</td>
<td style="border: 1px solid #ddd; padding: 8px;">29.9%</td>
</tr>
<tr>
<td style="border: 1px solid #ddd; padding: 8px;">Valet parking</td>
<td style="border: 1px solid #ddd; padding: 8px;">24.7%</td>
<td style="border: 1px solid #ddd; padding: 8px;">18.3%</td>
<td style="border: 1px solid #ddd; padding: 8px;">8.4%</td>
<td style="border: 1px solid #ddd; padding: 8px;">2.3%</td>
<td style="border: 1px solid #ddd; padding: 8px;">5.8%</td>
</tr>
</table>

**Key insights for investors:**

1. **Air conditioning** is standard across all categories — necessary
   but not differentiating.

2. **Interesting wine list** is the most discriminating facility —
   dropping from 84.4% in 3-Star restaurants to only 6.7% in
   Bib Gourmand. It is the strongest infrastructure signal
   of award potential.

3. **Bib Gourmand** intentionally rewards culinary excellence
   at accessible prices — lower facility rates in this category
   do not indicate lower quality, but a different evaluation criteria.

**Investor checklist — minimum facility benchmark for starred potential:**
- ✅ Air conditioning
- ✅ Interesting wine list
- ✅ Wheelchair access
- ✅ Car park

---

<h3 style="color: #28a745;">1.3 Facilities Count by Award Category</h3>

A final validation: does the total number of facilities
correlate with award level?

In [25]:
df['facilities_count'] = df['FacilitiesAndServices'].str.count(',') + 1
df['facilities_count'] = df['facilities_count'].fillna(0)

print(df.groupby('Award')['facilities_count'].mean().round(1).reindex(order_award))

Award
3 Stars                 3.8
2 Stars                 3.5
1 Star                  2.8
Bib Gourmand            2.0
Selected Restaurants    2.3
Name: facilities_count, dtype: float64


<h3 style="color: #28a745;">Finding: Quality Over Quantity</h3>

<table style="border-collapse: collapse; width: 100%;">
<tr style="background-color: #818181ff;">
<th style="border: 1px solid #ddd; padding: 8px;">Award</th>
<th style="border: 1px solid #ddd; padding: 8px;">Avg Facilities</th>
</tr>
<tr>
<td style="border: 1px solid #ddd; padding: 8px;"><span style="color: #dc3545;">3 Stars</span></td>
<td style="border: 1px solid #ddd; padding: 8px;">3.8</td>
</tr>
<tr style="background-color: #818181ff;">
<td style="border: 1px solid #ddd; padding: 8px;"><span style="color: #ffc107;">2 Stars</span></td>
<td style="border: 1px solid #ddd; padding: 8px;">3.5</td>
</tr>
<tr>
<td style="border: 1px solid #ddd; padding: 8px;"><span style="color: #28a745;">1 Star</span></td>
<td style="border: 1px solid #ddd; padding: 8px;">2.8</td>
</tr>
<tr style="background-color: #818181ff;">
<td style="border: 1px solid #ddd; padding: 8px;"><span style="color: #007bff;">Bib Gourmand</span></td>
<td style="border: 1px solid #ddd; padding: 8px;">2.0</td>
</tr>
<tr>
<td style="border: 1px solid #ddd; padding: 8px;"><span style="color: #6c757d;">Selected Restaurants</span></td>
<td style="border: 1px solid #ddd; padding: 8px;">2.3</td>
</tr>
</table>

**Key insight:** Selected Restaurants average more facilities
than Bib Gourmand (2.3 vs 2.0) — yet hold no Michelin recognition.

This suggests a critical distinction for investors:
**facility quantity does not substitute for facility quality.**
Selected Restaurants appear to prioritize infrastructure volume
over culinary excellence — the inverse of what Michelin evaluates.

Bib Gourmand demonstrates the correct hierarchy:
excellence in the plate first, infrastructure second.
Awarded restaurants build quantity on top of an already
recognized foundation of quality.

**Investor takeaway:** Investing in facilities before establishing
culinary excellence is a misallocation of capital.
The Michelin standard rewards the experience as a whole —
but it starts with what's on the plate.

---

## 2. Geographic Analysis

With the facility benchmark defined, the next question is:
**where should investors look?**

We analyze the geographic distribution of top-tier restaurants
across two levels: country and city.

Two metrics will be used:
- **Absolute frequency** — total 3-Star restaurants per location
- **Excellence density** — proportion of 3-Star restaurants
  relative to total restaurants in that location

In [5]:
# Absolute frequency — 3 Stars by country
stars_3_by_country=df[df['Award']=='3 Stars']['Country'].value_counts()
stars_3_by_country.head(10)

Country
France                 30
Japan                  19
Spain                  16
Italy                  15
USA                    14
Germany                11
United Kingdom         10
Hong Kong SAR China     7
Switzerland             4
Chinese Mainland        3
Name: count, dtype: int64

In [18]:
total_by_country=df['Country'].value_counts()

density = (stars_3_by_country/total_by_country*100).round(2)
density=density.sort_values(ascending=False).dropna()

print("=== Excellence Density by Country (%) ===")
print("(3-Star restaurants / total restaurants in country)\n")
density.head(15)

=== Excellence Density by Country (%) ===
(3-Star restaurants / total restaurants in country)



Country
Principality of Monaco    5.88
Norway                    4.17
China                     3.39
Hong Kong SAR China       3.20
Denmark                   1.98
Japan                     1.71
Slovenia                  1.43
Sweden                    1.32
Spain                     1.25
United Arab Emirates      1.16
Singapore                 1.03
France                    0.98
United Kingdom            0.90
Germany                   0.90
USA                       0.80
Name: count, dtype: float64

In [17]:
# Combine absolute and density into one table
geo_country = pd.DataFrame({
    'Total Restaurants': total_by_country,
    '3 Stars Count': stars_3_by_country,
    'Excellence Density (%)': density
}).dropna().sort_values('Excellence Density (%)', ascending=False)

print("=== Country Analysis — Absolute vs Density ===\n")
geo_country.head(15)

=== Country Analysis — Absolute vs Density ===



,Total Restaurants,3 Stars Count,Excellence Density (%)
Country,,,
Principality of Monaco,17,1.0,5.88
Norway,48,2.0,4.17
China,59,2.0,3.39
Hong Kong SAR China,219,7.0,3.20
Denmark,101,2.0,1.98
Japan,1114,19.0,1.71
Slovenia,70,1.0,1.43
Sweden,76,1.0,1.32
Spain,1285,16.0,1.25


### Finding: Absolute Volume vs Excellence Density

| Country | Total Restaurants | 3-Star Count | Density |
|---|---|---|---|
| Monaco | 17 | 1 | 5.88% |
| Norway | 48 | 2 | 4.17% |
| Hong Kong SAR China | 219 | 7 | 3.20% |
| Denmark | 101 | 2 | 1.98% |
| Japan | 1,114 | 19 | 1.71% |
| Spain | 1,285 | 16 | 1.25% |
| France | 3,055 | 30 | 0.98% |
| USA | 1,759 | 14 | 0.80% |

**Data note:** China's density (3.39%) may be slightly inflated —
some Macau restaurants were grouped under China due to
inconsistent location formatting in the original dataset.

**Key insights for investors:**

1. **France** leads in absolute count but density below 1% signals
   a saturated, highly competitive market.

2. **Japan and Spain** offer the best balance — large markets
   with above-average density. Most strategic for new investment.

3. **Hong Kong** stands out as a high-density market with
   meaningful volume — strong opportunity in Asian premium dining.

4. **Monaco and Norway** show high density but limited scale —
   niche markets with low competition but restricted growth potential.

**Strategic recommendation:** Japan, Spain, and Hong Kong offer
the strongest combination of market size and excellence density
for gastronomy investment.

---

### 2.2 City-Level Analysis

Country-level analysis tells investors **which markets to enter**.
City-level analysis tells them **exactly where to look**.

Applying the same two-metric approach — absolute count
and excellence density — at city level.

In [16]:
# Absolute frequency — 3 Stars by city
stars_3_by_city = df[df['Award'] == '3 Stars']['City'].value_counts()

# Excellence density by city
total_by_city = df['City'].value_counts()

geo_city = pd.DataFrame({
    'Total Restaurants': total_by_city,
    '3 Stars Count': stars_3_by_city,
}).dropna()

geo_city['Excellence Density (%)'] = (
    geo_city['3 Stars Count'] / geo_city['Total Restaurants'] * 100
).round(2)

geo_city = geo_city.sort_values('Excellence Density (%)', ascending=False)

print("=== City Analysis — Absolute vs Density ===\n")
geo_city.head(15)

=== City Analysis — Absolute vs Density ===



,Total Restaurants,3 Stars Count,Excellence Density (%)
City,,,
Brusaporto,1,1.0,100.0
El Puerto de Santa María,1,1.0,100.0
Dreis,1,1.0,100.0
Chagny,1,1.0,100.0
Kobarid,1,1.0,100.0
Grassau,1,1.0,100.0
Runate,1,1.0,100.0
Villaverde de Pontones,1,1.0,100.0
Perl,1,1.0,100.0


In [9]:
print(geo_city['Total Restaurants'].describe())

count     96.000000
mean      52.687500
std       98.250412
min        1.000000
25%        2.000000
50%        7.500000
75%       61.000000
max      532.000000
Name: Total Restaurants, dtype: float64


In [15]:
geo_city_filtered = geo_city[geo_city['Total Restaurants'] >= 20].sort_values(
    'Excellence Density (%)', ascending=False
)

print("=== City Analysis — Minimum 20 Restaurants ===\n")
geo_city_filtered.head(15)

=== City Analysis — Minimum 20 Restaurants ===



,Total Restaurants,3 Stars Count,Excellence Density (%)
City,,,
Donostia / San Sebastián,23,2.0,8.70
Marseille,28,2.0,7.14
Barcelona,93,4.0,4.30
Hamburg,53,2.0,3.77
San Diego,28,1.0,3.57
Oslo,28,1.0,3.57
Macau,59,2.0,3.39
Florence,30,1.0,3.33
Hong Kong,219,7.0,3.20


### Finding: City-Level Investment Hotspots

Filtering cities with at least 20 restaurants to ensure
market relevance for investment decisions.

| City | Total | 3-Star Count | Density |
|---|---|---|---|
| Donostia / San Sebastián | 23 | 2 | 8.70% |
| Marseille | 28 | 2 | 7.14% |
| Barcelona | 93 | 4 | 4.30% |
| Hamburg | 53 | 2 | 3.77% |
| Hong Kong | 219 | 7 | 3.20% |
| Tokyo | 532 | 11 | 2.07% |

**Key insights for investors:**

1. **Spain emerges as the strongest market** — San Sebastián leads
   city density at 8.70% while Barcelona combines high density
   with significant market size. This convergence at both country
   and city level makes Spain the top investment target.

2. **Hong Kong and Tokyo** confirm Asia as a strategic region —
   large markets with consistent excellence density above 2%.

3. **Hamburg and Marseille** are underrated European opportunities —
   strong density in mid-size markets with less competition
   than Paris or London.

**Combined recommendation:** Spain (San Sebastián + Barcelona)
and Asia (Hong Kong + Tokyo) represent the highest-confidence
investment targets based on geographic density analysis.

---

In [22]:
price_by_award=pd.crosstab(df['Award'],df['PriceLevel'],normalize='index').mul(100).round(1).reindex(order_award)
price_by_award.columns = ['Level 1', 'Level 2', 'Level 3', 'Level 4']

price_by_award

,Level 1,Level 2,Level 3,Level 4
Award,,,,
3 Stars,0.0,0.0,3.2,96.8
2 Stars,0.0,0.4,6.2,93.5
1 Star,0.1,2.8,34.4,62.7
Bib Gourmand,26.8,71.5,1.7,0.0
Selected Restaurants,5.4,42.3,41.3,11.0


### Finding: Price as a Commitment Signal

| Award | Level 1 | Level 2 | Level 3 | Level 4 |
|---|---|---|---|---|
| 3 Stars | 0.0% | 0.0% | 3.2% | 96.8% |
| 2 Stars | 0.0% | 0.4% | 6.2% | 93.5% |
| 1 Star | 0.1% | 2.8% | 34.4% | 62.7% |
| Bib Gourmand | 26.8% | 71.5% | 1.7% | 0.0% |
| Selected Restaurants | 5.4% | 42.3% | 41.3% | 11.0% |

**Key insights for investors:**

1. **96.8% of 3-Star restaurants operate at Level 4 pricing** —
   premium pricing is not optional at the highest tier.
   It reflects the complete experience: location, facilities,
   service, and culinary excellence combined.

2. **Bib Gourmand confirms the exception** — 98.3% at Levels 1-2.
   This category is specifically designed to reward excellence
   at accessible prices. A different investment thesis entirely.

3. **Selected Restaurants expose a critical insight** — 11% charge
   Level 4 prices but hold no award. Premium pricing alone does
   not guarantee recognition. The Michelin standard evaluates
   the complete experience, not just the price tag.

**Investor takeaway:** Price Level 4 is a necessary condition
for 3-Star potential — but it is not sufficient on its own.
It must be backed by the right facilities, location, and culinary
excellence to translate into Michelin recognition.

---

### 4.2 Average Price Level by Award Category

A single metric summarizing the price positioning of each award tier.
This gives investors a quick benchmark for expected investment level.

In [23]:
avg_price_by_award=df.groupby('Award')['PriceLevel'].mean().round(2).reindex(order_award)
avg_price_by_award

Award
3 Stars                 3.97
2 Stars                 3.93
1 Star                  3.60
Bib Gourmand            1.75
Selected Restaurants    2.58
Name: PriceLevel, dtype: float64

### Finding: The Price-Award Correlation

| Award | Average Price Level |
|---|---|
| 3 Stars | 3.97 |
| 2 Stars | 3.93 |
| 1 Star | 3.60 |
| Bib Gourmand | 1.75 |
| Selected Restaurants | 2.58 |

**Key insights for investors:**

1. **Starred restaurants form a distinct price cluster** — average
   above 3.60, converging toward 4.0 at the top tiers.
   The gap between 3-Star (3.97) and 2-Star (3.93) is minimal —
   both tiers operate at near-maximum pricing.

2. **Selected Restaurants reveal a value trap** — average price
   of 2.58, higher than Bib Gourmand (1.75) yet unrecognized
   by Michelin. These restaurants charge more but deliver less
   perceived value than awarded competitors at lower price points.

3. **Bib Gourmand is the outlier by design** — low price,
   high recognition. A completely different investment thesis
   focused on volume and accessibility rather than premium experience.

**Investor takeaway:** If the goal is Michelin Star recognition,
budget for Level 4 pricing from day one. Anything below that
signals a positioning inconsistent with starred potential.

---

### 4.3 Price Level in Strategic Markets

We identified Spain, Japan, and Hong Kong as the top investment
targets based on geographic density analysis.

The final question is: **what is the price positioning of
3-Star restaurants in these specific markets?**

This gives investors a concrete financial benchmark
for each target market.

In [24]:
# Filter strategic markets — 3 Stars only
strategic_markets = ['Spain', 'Japan', 'Hong Kong SAR China']

strategic_price=df[(df['Award'] == '3 Stars') & (df['Country'].isin(strategic_markets))
                   ].groupby('Country')['PriceLevel'].agg(['mean','min','max']).round(2)

strategic_price.columns = ['Average', 'Min', 'Max']
strategic_price=strategic_price.reindex(strategic_markets)

strategic_price

,Average,Min,Max
Country,,,
Spain,4.00,4,4
Japan,3.89,3,4
Hong Kong SAR China,3.86,3,4


### Finding: Price Positioning in Strategic Markets

| Country | Average | Min | Max |
|---|---|---|---|
| Spain | 4.00 | 4 | 4 |
| Japan | 3.89 | 3 | 4 |
| Hong Kong SAR China | 3.86 | 3 | 4 |

**Key insights for investors:**

1. **Spain operates exclusively at Level 4** — no exceptions.
   The Spanish fine dining scene is built around modern,
   sophisticated experiences that command maximum pricing.

2. **Japan and Hong Kong show Level 3 outliers** — reflecting
   a cultural tradition of excellence in more intimate,
   heritage-driven restaurants. Family-owned establishments
   and traditional culinary formats (omakase, kaiseki)
   can achieve 3-Star recognition without maximum pricing.
   This opens an investment window at a slightly lower ticket.

3. **Cross-referencing with facilities confirms this pattern** —
   Japanese 3-Star restaurants uniquely feature `Shoes must be removed`,
   signaling authentic cultural environments that prioritize
   culinary tradition over modern infrastructure.

**Investor takeaway:** Spain demands full Level 4 commitment.
Japan and Hong Kong offer rare Level 3 opportunities —
but only in culturally-rooted formats with deep culinary heritage.

---

## 5. Investment Profile Summary

Based on the three analytical pillars — facilities, geography,
and price — we can now define the complete profile of a
3-Star Michelin restaurant for investment decision-making.

This summary consolidates all findings into an actionable
framework for the investment committee.

---

### 5.1 The 3-Star Michelin Investment Blueprint

**Infrastructure Requirements**
| Facility | Penetration in 3-Star Restaurants |
|---|---|
| Air conditioning | 93.5% — mandatory |
| Interesting wine list | 84.4% — strongest differentiator |
| Wheelchair access | 56.5% — expected |
| Car park | 48.1% — recommended |
| Valet parking | 24.7% — premium signal |

**Average facilities per award tier**
| Award | Avg Facilities |
|---|---|
| 3 Stars | 3.8 |
| 2 Stars | 3.5 |
| 1 Star | 2.8 |
| Bib Gourmand | 2.0 |
| Selected Restaurants | 2.3 |

**Price Positioning**
- Level 4 is the standard across 97% of 3-Star restaurants
- Level 3 exists only in culturally-rooted Asian markets
- Premium pricing reflects experience, not just food quality

**Target Markets**
| Market | 3-Star Count | Excellence Density | Price Standard |
|---|---|---|---|
| Spain | 16 | 1.25% | Level 4 only |
| Japan | 19 | 1.71% | Level 3–4 |
| Hong Kong | 7 | 3.20% | Level 3–4 |

---

### 5.2 Investor Decision Framework

A restaurant worth investing in should meet the following criteria:

✅ **Price:** Operating at Level 3 minimum — Level 4 preferred  
✅ **Wine:** Interesting wine list present  
✅ **Access:** Wheelchair access and car park available  
✅ **Location:** Based in a high-density excellence market  
✅ **Market:** Spain, Japan, or Hong Kong as primary targets  
✅ **Foundation:** Culinary excellence established before
   scaling infrastructure investment

⚠️ **Red flags — reconsider investment if:**  
❌ No registered facilities  
❌ Price Level 1 or 2 with no Bib Gourmand positioning  
❌ Located in a low-density market with no regional cluster  
❌ High facility count without Michelin recognition —
   signals quantity over quality misallocation

---

### 5.3 Key Takeaway for the Investment Committee

> *"A 3-Star Michelin restaurant is not just a restaurant —
> it is a complete experience. The data shows that excellence
> at this level requires a deliberate combination of
> premium pricing, curated infrastructure, and strategic
> location in proven high-density markets."*
>
> *"Facility quantity does not substitute for culinary excellence.
> The data shows that Selected Restaurants — unrecognized
> despite higher prices and more facilities than Bib Gourmand —
> represent a cautionary tale: investing in infrastructure
> before establishing excellence is a misallocation of capital."*
>
> *"Investing without validating these three pillars
> significantly increases the risk of backing an establishment
> that charges premium prices without delivering
> the experience that Michelin — and its clientele — demands."*

---